<a href="https://colab.research.google.com/github/Sarumathi1/cdw/blob/Day5/Day5task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import GPT2Tokenizer, TextDataset, DataCollatorForLanguageModeling, GPT2LMHeadModel, pipeline, \
                         Trainer, TrainingArguments

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')  # load up a standard gpt2 model

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [7]:
pds_data = TextDataset(
    tokenizer=tokenizer,
    file_path='/content/book.txt',  # Principles of Data Science - Sinan Ozdemir
    block_size=64  # length of each chunk of text to use as a datapoint
)

/usr/local/lib/python3.11/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [8]:
pds_data[0], pds_data[0].shape  # inspect the first point

(tensor([   44,  7156,  1404, 25994,   198,    44,  7156,  1404, 25994,   198,
            51, 25994,    45, 43781,  3268,   362,    46,    20,    46,   198,
            51, 25994,    45, 43781,  3268, 32215,   198, 42131,   416,   198,
            35, 43664,  3698,  8782, 15154, 34509, 42131,   416,   198,    35,
         43664,  3698,  8782, 15154, 34509,   198, 30650,   198,   198, 24492,
           739,  8568, 17098,   422,   383, 36619,   416,   198, 37046, 13661,
         12052,   198,    18,  6479]),
 torch.Size([64]))

In [9]:
print(tokenizer.decode(pds_data[0]))

MEGATECH
MEGATECH
TECHNOLOGY IN 2O5O
TECHNOLOGY IN 2050
edited by
DANIEL FRANKLINedited by
DANIEL FRANKLIN
Books

Published under exclusive licence from The Economist by
Profile Books Ltd
3 Hol


In [10]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False,
    # MLM is Masked Language Modelling (for BERT + auto-encoding tasks)
)

In [11]:
# example of how collator pads data dynamically
collator_example = data_collator([tokenizer('I am an input'), tokenizer('So am I')])

collator_example

{'input_ids': tensor([[   40,   716,   281,  5128],
        [ 2396,   716,   314, 50256]]), 'attention_mask': tensor([[1, 1, 1, 1],
        [1, 1, 1, 0]]), 'labels': tensor([[  40,  716,  281, 5128],
        [2396,  716,  314, -100]])}

In [12]:
collator_example.input_ids  # 50256 is our pad token id

tensor([[   40,   716,   281,  5128],
        [ 2396,   716,   314, 50256]])

In [13]:
tokenizer.pad_token_id

50256

In [14]:
collator_example.attention_mask  # Note the 0 in the attention mask where we have a pad token

tensor([[1, 1, 1, 1],
        [1, 1, 1, 0]])

In [15]:
collator_example.labels  # note the -100 to ignore loss calculation for the padded token
# Labels are shifted inside the GPT model so we don't need to worry about that

tensor([[  40,  716,  281, 5128],
        [2396,  716,  314, -100]])

In [16]:
model = GPT2LMHeadModel.from_pretrained('gpt2')  # load up a GPT2 model

pretrained_generator = pipeline(  # create a generator with built in params
    'text-generation', model=model, tokenizer='gpt2',
    config={'max_length': 200, 'do_sample': True, 'top_p': 0.9, 'temperature': 0.7, 'top_k': 10}
)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cpu


In [17]:
print('----------')
for generated_sequence in pretrained_generator('This dataset shows the relationship', num_return_sequences=3):
    print(generated_sequence['generated_text'])
    print('----------')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


----------
This dataset shows the relationship between each individual's diet of foods ranging from low-fat to high-fat to high-glycemic, total, and whole grains and fruit and vegetables over time into individual years. To date, the largest cohort of Americans
----------
This dataset shows the relationship between total and total mortality:

As it can be done from the last article in this series – this is what the data reads like in the above table. It is based on mortality rates based on the number of deaths
----------
This dataset shows the relationship between testosterone levels and serum testosterone levels in young men with early onset hypertension. There is an almost complete genetic correlation between testosterone levels measured within the first trimester of age and serum testosterone levels in later development. The data show that
----------


In [19]:
training_args = TrainingArguments(
    output_dir="./gpt2_pds", #The output directory
    overwrite_output_dir=True, #overwrite the content of the output directory
    num_train_epochs=3, # number of training epochs
    per_device_train_batch_size=32, # batch size for training
    per_device_eval_batch_size=32,  # batch size for evaluation
    logging_steps=10,
    load_best_model_at_end=True,
    evaluation_strategy='epoch',
    save_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=pds_data.examples[:int(len(pds_data.examples)*.8)],
    eval_dataset=pds_data.examples[int(len(pds_data.examples)*.8):]
)

trainer.evaluate()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


{'eval_loss': 4.366634368896484,
 'eval_model_preparation_time': 0.0028,
 'eval_runtime': 12.2521,
 'eval_samples_per_second': 2.04,
 'eval_steps_per_second': 0.082}

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,No log,3.868581,0.002800
2,No log,3.800093,0.002800
3,4.269800,3.783803,0.002800


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=12, training_loss=4.2320709228515625, metrics={'train_runtime': 456.5012, 'train_samples_per_second': 0.637, 'train_steps_per_second': 0.026, 'total_flos': 9504497664000.0, 'train_loss': 4.2320709228515625, 'epoch': 3.0})

In [21]:
trainer.evaluate()  # loss decrease is slowing down so we are hitting our limit

{'eval_loss': 3.7838029861450195,
 'eval_model_preparation_time': 0.0028,
 'eval_runtime': 8.8529,
 'eval_samples_per_second': 2.824,
 'eval_steps_per_second': 0.113,
 'epoch': 3.0}

In [22]:
trainer.save_model()

In [23]:
loaded_model = GPT2LMHeadModel.from_pretrained('./gpt2_pds')

finetuned_generator = pipeline(
    'text-generation', model=loaded_model, tokenizer=tokenizer,
    config={'max_length': 200, 'do_sample': True, 'top_p': 0.9, 'temperature': 0.7, 'top_k': 10}
)

Device set to use cpu


In [24]:
# examples are now sustainably about data
print('----------')
for generated_sequence in finetuned_generator('what is megatech', num_return_sequences=3):
    print(generated_sequence['generated_text'])
    print('----------')

----------
what is megatech?" asked John Faubert, in an effort to provide some common understanding of the subject of gender differences.
The gender equality movement will continue, the debate has been riled by a spate of recent victories, such as
----------
what is megatech?"
With more than half a century ago, it was clear that the internet allowed anyone to connect the two worlds. Now, Google is taking a different tack by offering its services on a much larger scale. As it prepares
----------
what is megatech] that you are here to do? And do you mean to help this project go forward?
I think I should say something about my attitude about it. In 2006 I came across the book A Short Description of Social Science
----------
